# Lab 21 — Thân Tiến Đạt · 2A202603023



In [1]:
# @title 1. Setup — upload gói ZIP + cài dependency
import os, pathlib, subprocess, sys, zipfile

SOURCE = "upload"  # @param ["upload", "github"]
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
REPO = "https://github.com/Datbadboiz11/Day21-ThanTienDat-2A202603023-Track3-Finetuning-Lab.git"
PROJECT = pathlib.Path("/content/lab21_2A202603023")

if not (PROJECT / "requirements.txt").exists():
    if SOURCE == "upload":
        from google.colab import files
        print("Chọn COLAB_INPUT_2A202603023.zip (hoặc ZIP kết quả để tiếp tục).")
        uploaded = files.upload()
        archives = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise RuntimeError("Hãy tải lên đúng một file ZIP của dự án.")
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                destination = (pathlib.Path("/content") / member.filename).resolve()
                if not destination.is_relative_to(PROJECT) or "\\" in member.filename:
                    raise RuntimeError("ZIP có cấu trúc không đúng; dùng gói do export_colab.py tạo.")
            archive.extractall("/content")
    else:
        subprocess.run(["git", "clone", REPO, str(PROJECT)], check=True)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
for name in ("results", "adapters"):
    (PROJECT / name).mkdir(exist_ok=True)
required = ["data/train_seed.jsonl", "data/eval_target.jsonl", "data/eval_regression.jsonl", "data/checksums.json"]
missing = [name for name in required if not (PROJECT / name).is_file()]
if missing:
    raise RuntimeError(f"Gói ZIP thiếu dữ liệu: {missing}")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
os.environ["COMPUTE_TIER"] = "T4"
os.environ["EPOCHS"] = "2"
os.environ["MASK_MODE"] = "assistant-only"
for name in ("EVAL_LIMIT", "BASE_MODEL", "ONLY", "FORCE_RETRAIN", "LAB_BACKUP_DIR"):
    os.environ.pop(name, None)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chọn Runtime > Change runtime type > T4 GPU, rồi chạy lại ô Setup.")
print("GPU:", torch.cuda.get_device_name(0))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["LAB_BACKUP_DIR"] = "/content/drive/MyDrive/Lab21_2A202603023"
    print("Sao lưu sau mỗi NB và mỗi adapter vào", os.environ["LAB_BACKUP_DIR"])

def run_stages(*stages):
    subprocess.run([sys.executable, "-u", "scripts/colab_run.py", *stages], check=True)


Chọn COLAB_INPUT_2A202603023.zip (hoặc ZIP kết quả để tiếp tục).


Saving COLAB_INPUT_2A202603023.zip to COLAB_INPUT_2A202603023.zip
GPU: Tesla T4
Mounted at /content/drive
Sao lưu sau mỗi NB và mỗi adapter vào /content/drive/MyDrive/Lab21_2A202603023


In [2]:
# @title 2. Kiểm tra môi trường, dữ liệu và unit test
subprocess.run([sys.executable, "scripts/verify.py", "--smoke"], check=True)
import json, importlib.metadata, datetime
versions = {}
for package in ("torch", "transformers", "trl", "peft", "accelerate", "datasets", "bitsandbytes", "torchao"):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = "not installed"
environment = {"utc_time": datetime.datetime.now(datetime.timezone.utc).isoformat(),
               "gpu": torch.cuda.get_device_name(0),
               "vram_total_gb": torch.cuda.get_device_properties(0).total_memory / 1024**3,
               "python": sys.version, "versions": versions,
               "tier": os.environ["COMPUTE_TIER"], "epochs": os.environ["EPOCHS"],
               "mask_mode": os.environ["MASK_MODE"], "eval_limit": None}
(PROJECT / "results/environment.json").write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8")
lock = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
(PROJECT / "results/requirements-lock.txt").write_text(lock, encoding="utf-8")
print(json.dumps(environment, ensure_ascii=False, indent=2))


{
  "utc_time": "2026-10-07T08:04:48.807829+00:00",
  "gpu": "Tesla T4",
  "vram_total_gb": 14.56317138671875,
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "5.18.0",
    "trl": "1.14.2",
    "peft": "0.21.1",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
    "bitsandbytes": "0.50.2",
    "torchao": "0.18.0"
  },
  "tier": "T4",
  "epochs": "2",
  "mask_mode": "assistant-only",
  "eval_limit": null
}


In [3]:
# @title 3. NB1 + NB2 — chứng minh mask và đóng băng baseline
if (PROJECT / "adapters/correct/adapter_model.safetensors").exists():
    print("Đã có adapter: giữ nguyên baseline; không đo lại sau khi đã train.")
else:
    run_stages("nb1", "nb2")
frozen = json.loads((PROJECT / "results/baselines_frozen.json").read_text(encoding="utf-8"))
print("(a) =", frozen["baseline_a"]["target"], "· (b) =", frozen["baseline_b"]["target"])
if frozen["baseline_b"]["target"] <= frozen["baseline_a"]["target"]:
    raise RuntimeError("Dừng trước khi train: prompt (b) chưa thắng (a). Cải thiện prompt và đo lại NB2 trước NB3.")
print("Đọc mốc (b) này trước khi chạy ô 4. Không sửa eval hoặc prompt (b) sau khi train.")


(a) = 0.0 · (b) = 0.765
Đọc mốc (b) này trước khi chạy ô 4. Không sửa eval hoặc prompt (b) sau khi train.


In [4]:
# @title 4. NB3 + NB4 — train đúng và ba đối chứng
frozen = json.loads((PROJECT / "results/baselines_frozen.json").read_text(encoding="utf-8"))
if frozen.get("smoke_mode") or frozen["baseline_b"]["target"] <= frozen["baseline_a"]["target"]:
    raise RuntimeError("Hoàn thành baseline toàn bộ và cải thiện (b) ở ô 3 trước khi train.")
if (PROJECT / "adapters/correct/adapter_model.safetensors").exists():
    print("Giữ adapter correct đã hoàn thành.")
else:
    run_stages("nb3")
run_stages("nb4")  # NB4 tự bỏ qua các adapter đã lưu


In [5]:
# @title 5. NB5 — đánh giá và sinh bảng số liệu báo cáo
run_stages("nb5")
subprocess.run([sys.executable, "scripts/build_measured_report.py"], check=True)
from IPython.display import Markdown, display
display(Markdown((PROJECT / "submission/MEASURED_RESULTS.md").read_text(encoding="utf-8")))
from labkit import report
report.backup_progress(PROJECT)


# Số liệu thực nghiệm — Lab 21

Học viên: **Thân Tiến Đạt** · MSSV: **2A202603023**

Tài liệu này được sinh từ kết quả thật. Phân tích và phản tư cá nhân nằm trong REPORT.md.

Chỉ đo được 1 ca FT thua (b). Chưa đủ 2 ca thua theo rubric; ghi đúng thực tế, không gán ca hoà thành thua.

## Cấu hình đã chạy

Tier: `T4` · Model: `unsloth/Qwen3.5-4B` · Target: 50 mẫu · Regression: 15 mẫu.

GPU thực tế: Tesla T4.

## Bằng chứng mask và token

Nguồn: results/mask_proof.json, template_check.json, token_stats.json.

| chỉ số | giá trị |
|---|---|
| mask_mode | assistant-only |
| n_supervised | 39 |
| n_total | 94 |
| supervised_fraction | 0.4149 |
| answer_is_supervised | True |
| question_is_masked | True |

Đoạn được tính loss:

```text
</think>

{"intent": "doi_tra", "urgency": "trung_binh", "product": "balo laptop", "sentiment": "trung_tinh"}<|im_end|>

```

Template:

```json
{
  "ok": true,
  "open_tag_present": true,
  "body_present": true,
  "rendered": "<|im_start|>user\n2+2?<|im_end|>\n<|im_start|>assistant\n<think>\nbuoc 1: kiem tra. buoc 2: tra loi.\n</think>\n\n4<|im_end|>\n",
  "verdict": "reasoning preserved — safe to train on traces"
}
```

Thống kê token:

```json
{
  "n": 250,
  "mean": 93.1,
  "p50": 93,
  "p95": 98,
  "p99": 100,
  "max": 101,
  "suggested_max_length": 256
}
```

## So sánh ba baseline

Nguồn: results/verdict.json; baseline đóng băng tại NB2.

| run | target | regression | format | latency_ms | n |
|---|---|---|---|---|---|
| (a) base + naive prompt | 0.0 | 0.7911 | 0.0 | 3400.3 | 50 |
| (b) base + optimized prompt | 0.765 | 0.7911 | 1.0 | 1063.7 | 50 |
| (c) LoRA fine-tune | 0.97 | 0.6111 | 1.0 | 1409.2 | 50 |

## Bốn cấu hình huấn luyện

Nguồn: results/runs.csv và results/autopsy.json.

| run | placement | r | trainable_params | learning_rate | max_steps | final_loss | target | format | train_seconds | peak_vram_gb |
|---|---|---|---|---|---|---|---|---|---|---|
| correct | text-linear | 16 | 32464896 | 0.0001 | 30 | 0.6268 | 0.97 | 1.0 | 415.6 | 8.78 |
| attn_only | attn-only | 283 | 32456704 | 0.0001 | 30 | 0.5366 | 0.97 | 1.0 | 276.8 | 8.79 |
| wrong_lr | text-linear | 16 | 32464896 | 1e-05 | 30 | 1.5702 | 0.0 | 0.0 | 405.1 | 8.78 |
| qlora | text-linear | 16 | 32464896 | 0.0001 | 30 | 0.7058 | 0.94 | 1.0 | 474.1 | 3.86 |

Lịch sử loss: results/training_log_<run>.json. Xếp hạng bằng target, không bằng train loss.

## Cổng hồi quy: FAILED

```json
{
  "passed": false,
  "reasons": [
    "general capability regressed by 0.180 (tolerance 0.020). See deck §6.3 — add 1-5% replay data."
  ],
  "target_delta": 0.20499999999999996,
  "regression_delta": -0.18000000000000005
}
```

## Ví dụ đối chiếu với (b)

33 ca thắng, 1 ca thua, 16 ca hoà. Nguồn: results/qualitative.json.

| i | ticket | label | baseline_b_pred | ft_pred | base_score | ft_score | outcome |
|---|---|---|---|---|---|---|---|
| 26 | Alo shop, mình đặt máy xay sinh tố mã đơn VN724342. Trả lại tiền. Không vội. Nhờ shop kiểm tra. | {"intent": "hoan_tien", "urgency": "thap", "product": "máy xay sinh tố", "sentiment": "trung_tinh"} | {"intent": "hoan_tien", "urgency": "thap", "product": "máy xay sinh tố", "sentiment": "trung_tinh"} | {"intent": "doi_tra", "urgency": "thap", "product": "máy xay sinh tố", "sentiment": "trung_tinh"} | 1.0 | 0.75 | loss |
| 36 | Chào shop, mình đặt tai nghe bluetooth mã đơn VN161530. Giá bao nhiêu. Hỏi cho biết thôi. Rất thất vọng. | {"intent": "hoi_thong_tin", "urgency": "thap", "product": "tai nghe bluetooth", "sentiment": "tieu_cuc"} | {"intent": "doi_tra", "urgency": "trung_binh", "product": "tai nghe bluetooth", "sentiment": "tieu_cuc"} | {"intent": "hoi_thong_tin", "urgency": "thap", "product": "tai nghe bluetooth", "sentiment": "tieu_cuc"} | 0.5 | 1.0 | win |
| 49 | Chào shop, mình đặt ốp lưng điện thoại mã đơn VN833689. Sai màu. Sớm nhé. Shop xem giúp. | {"intent": "san_pham_loi", "urgency": "trung_binh", "product": "ốp lưng điện thoại", "sentiment": "trung_tinh"} | {"intent": "san_pham_loi", "urgency": "cao", "product": "ốp lưng điện thoại", "sentiment": "tieu_cuc"} | {"intent": "san_pham_loi", "urgency": "trung_binh", "product": "ốp lưng điện thoại", "sentiment": "trung_tinh"} | 0.5 | 1.0 | win |
| 2 | Xin chào, mình đặt đèn bàn LED mã đơn VN880807. Hoàn tiền. Quá hạn rồi. Cảm ơn shop nhiều. | {"intent": "hoan_tien", "urgency": "cao", "product": "đèn bàn LED", "sentiment": "tich_cuc"} | {"intent": "hoan_tien", "urgency": "cao", "product": "đèn bàn LED", "sentiment": "tich_cuc"} | {"intent": "hoan_tien", "urgency": "cao", "product": "đèn bàn LED", "sentiment": "tich_cuc"} | 1.0 | 1.0 | tie |
| 3 | Cho mình hỏi, mình đặt bình giữ nhiệt mã đơn VN804124. Chưa thấy tiền. Khi nào tiện. Cảm ơn shop nhiều. | {"intent": "hoan_tien", "urgency": "thap", "product": "bình giữ nhiệt", "sentiment": "tich_cuc"} | {"intent": "hoan_tien", "urgency": "trung_binh", "product": "bình giữ nhiệt", "sentiment": "tich_cuc"} | {"intent": "hoan_tien", "urgency": "trung_binh", "product": "bình giữ nhiệt", "sentiment": "tich_cuc"} | 0.75 | 0.75 | tie |

## Phần cần viết trong REPORT.md

Giải thích các đối chứng (mỗi câu hỏi ít nhất 3 câu), diễn giải phán quyết ≥100 từ, kết luận ≥150 từ và phản tư cá nhân. Không thay đổi eval hay ngưỡng chấm để tạo kết quả đẹp.


In [6]:
# @title 6. Kiểm tra và tải ZIP kết quả (cũng chạy được sau khi một ô bị lỗi)
from google.colab import files
verification = subprocess.run([sys.executable, "scripts/verify.py"])
if verification.returncode:
    print("Chưa sẵn sàng nộp. Báo cáo cần được viết từ kết quả thật; xem các FAIL ở trên.")
output = pathlib.Path("/content/LAB21_RESULTS_2A202603023.zip")
subprocess.run([sys.executable, "scripts/export_colab.py", "--mode", "progress", "--output", str(output)], check=True)
files.download(str(output))
print("Gửi ZIP kết quả để hoàn thiện phân tích REPORT.md. NB6 là tuỳ chọn.")
